**Data Lead**

Input: Raw .wav, .jpg, and .txt files from the 00_Raw_Data folder.

Output: Renamed/cleaned files in 01_Processed_Data and the metadata.csv.

Goal: To be the "Single Source of Truth." If it’s not in Data Lead's CSV, the rest of the team doesn't touch it.

Run all code below after data collection is complete

In [ ]:
# 1. Connect to Google Drive & Imports
from google.colab import drive
import os, shutil
from datetime import datetime

drive.mount('/content/drive')

# 2. Define the exact project paths (Cell 1 is canonical — all sub-paths live here)
PROJECT_ROOT = "/content/drive/MyDrive/DL_Project_2026"

RAW_DIR = os.path.join(PROJECT_ROOT, "00_Raw_Data")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "01_Processed_Data")
EMBED_DIR = os.path.join(PROJECT_ROOT, "02_Embeddings")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "03_Final_Results")
VIS_DIR = os.path.join(PROJECT_ROOT, "04_Visual_Checks")

NOTEBOOK_ID = "01"

print(f"✅ Connection successful! Project folder located at: {PROJECT_ROOT}")

# --- 3. THE ARCHIVE UTILITY ---
# Archive format: 99_Versions/{notebook_id}_{timestamp}_{folder_name}/
# (Directive 3: archive before write; use {notebook_id}_{timestamp} so archives are
# attributable to the notebook that produced them. The _{folder_name} suffix keeps
# snapshots distinct when a single notebook archives more than one directory in one run.)
def archive_previous_run(directory_path, notebook_id):
    if os.path.exists(directory_path) and os.listdir(directory_path):
        version_root = os.path.join(PROJECT_ROOT, "99_Versions")
        os.makedirs(version_root, exist_ok=True)

        timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
        folder_name = os.path.basename(directory_path)
        archive_name = os.path.join(
            version_root, f"{notebook_id}_{timestamp}_{folder_name}"
        )

        print(f"📦 Archiving previous results to: {archive_name}")
        shutil.move(directory_path, archive_name)
    os.makedirs(directory_path, exist_ok=True)

# --- 4. EXECUTE ARCHIVE FOR NOTEBOOK 01 ---
# This notebook writes to PROCESSED_DIR (and later VIS_DIR for visual checks).
archive_previous_run(PROCESSED_DIR, notebook_id=NOTEBOOK_ID)

# Re-create subfolders for the fresh run
for sub in ['audio', 'images', 'lyrics']:
    os.makedirs(os.path.join(PROCESSED_DIR, sub), exist_ok=True)


# --- 5. DATA SANITY CHECK ---
# Scans 00_Raw_Data to ensure the 150 tracks are named correctly.
audio_folder = os.path.join(RAW_DIR, "audio")
image_folder = os.path.join(RAW_DIR, "images")
lyrics_folder = os.path.join(RAW_DIR, "lyrics")

AUDIO_EXTS = ('.mp3', '.wav', '.m4a', '.flac', '.ogg')

if not all(os.path.exists(d) for d in [audio_folder, image_folder, lyrics_folder]):
    print("❌ ERROR: Raw data folders are missing!")
else:
    audio_files = [
        f for f in os.listdir(audio_folder)
        if not f.startswith('.') and f.lower().endswith(AUDIO_EXTS)
    ]
    missing_images, missing_lyrics = [], []

    print(f"🧐 Scanning {len(audio_files)} audio files for multimodal matches...")

    for f in audio_files:
        base_name = os.path.splitext(f)[0]
        has_image = any(os.path.exists(os.path.join(image_folder, base_name + ext)) for ext in ['.jpg', '.jpeg', '.png'])
        if not has_image: missing_images.append(f)

        has_lyrics = os.path.exists(os.path.join(lyrics_folder, base_name + ".txt"))
        if not has_lyrics: missing_lyrics.append(f)

    print("-" * 30)
    if not missing_images and not missing_lyrics:
        print(f"✅ DATA INTEGRITY VERIFIED: All {len(audio_files)} tracks are ready.")
    else:
        if missing_images:
            print(f"❌ MISSING IMAGES ({len(missing_images)}):")
            for m in missing_images: print(f"  - {m}")
        if missing_lyrics:
            print(f"\n❌ MISSING LYRICS ({len(missing_lyrics)}):")
            for m in missing_lyrics: print(f"  - {m}")
        print("\n⚠️ ACTION REQUIRED: Fix filenames in Drive before proceeding!")


Mounted at /content/drive
✅ Connection successful! Project folder located at: /content/drive/MyDrive/DL_Project_2026
❌ ERROR: Raw data folders are missing!


In [ ]:
# --- STEP 1: INSTALL TOOLS ---
!pip install pydub
!apt-get install ffmpeg -y

import os
import numpy as np
import pandas as pd
from pydub import AudioSegment
from PIL import Image

# Ensure the "Processed" subfolders exist (Cell 2 already makes these; defensive no-op here).
for sub in ['audio', 'images', 'lyrics']:
    os.makedirs(os.path.join(PROCESSED_DIR, sub), exist_ok=True)

# --- STEP 2: LOAD THE RICH CSV ---
# Required 7-column schema: file_name, song_title, artist_name, genre,
# emotion_category, emotion_subcategory, contributor
# Let the read propagate on failure — the manifest cannot be built without labels.
labels_path = os.path.join(RAW_DIR, "raw_data_labels.csv")
labels_df = pd.read_csv(labels_path)
labels_df['file_name'] = labels_df['file_name'].str.strip()

REQUIRED_LABEL_COLS = {"file_name", "song_title", "artist_name", "genre",
                       "emotion_category", "emotion_subcategory", "contributor"}
missing_cols = REQUIRED_LABEL_COLS - set(labels_df.columns)
assert not missing_cols, f"raw_data_labels.csv missing required columns: {missing_cols}"
print(f"📖 Loaded rich labels for {len(labels_df)} tracks.")

# --- STEP 3: THE TRANSFORMATION LOOP ---
AUDIO_EXTS = ('.mp3', '.wav', '.m4a', '.flac', '.ogg')
MIN_SOURCE_MS = 60_000        # need ≥60s of source to cut 30–60s out
EXPECTED_CLIP_MS = 30_000     # clip length

audio_raw_path = os.path.join(RAW_DIR, "audio")
image_raw_path = os.path.join(RAW_DIR, "images")
lyrics_raw_path = os.path.join(RAW_DIR, "lyrics")

data_log = []
files = sorted([
    f for f in os.listdir(audio_raw_path)
    if not f.startswith('.') and f.lower().endswith(AUDIO_EXTS)
])

print(f"🚀 Processing {len(files)} tracks...")

for i, filename in enumerate(files):
    track_id = f"track_{i+1:03}"
    produced = []  # track files we've written in this iteration for atomic cleanup

    try:
        # A. Audio: Trim 30s–60s & convert to WAV (validate lengths so silent clips
        # cannot silently slip through and poison downstream embeddings).
        src_audio_path = os.path.join(audio_raw_path, filename)
        audio = AudioSegment.from_file(src_audio_path)
        if len(audio) < MIN_SOURCE_MS:
            raise ValueError(
                f"{filename}: source is {len(audio)/1000:.1f}s, need ≥{MIN_SOURCE_MS/1000}s"
            )
        clip = audio[30_000:60_000]
        if len(clip) < EXPECTED_CLIP_MS - 100:  # allow 100ms jitter
            raise ValueError(f"{filename}: trimmed clip is only {len(clip)} ms")
        samples = np.array(clip.get_array_of_samples())
        if samples.size == 0 or np.abs(samples).mean() < 1.0:
            raise ValueError(f"{filename}: trimmed clip appears silent (mean abs ≈ 0)")

        wav_out = os.path.join(PROCESSED_DIR, "audio", f"{track_id}.wav")
        clip.export(wav_out, format="wav")
        produced.append(wav_out)

        # B. Images: Resize to 224x224 RGB. Hard-fail if missing — every manifest
        # row must have its JPG on disk (row N of any .npy = track_id N).
        base_name = filename.rsplit('.', 1)[0]
        img_file = None
        for ext in ['.jpg', '.jpeg', '.png']:
            path = os.path.join(image_raw_path, base_name + ext)
            if os.path.exists(path):
                img_file = path
                break
        if img_file is None:
            raise FileNotFoundError(f"{filename}: no image found in {image_raw_path}")

        img_out = os.path.join(PROCESSED_DIR, "images", f"{track_id}.jpg")
        Image.open(img_file).convert('RGB').resize((224, 224)).save(img_out)
        produced.append(img_out)

        # C. Lyrics: Clean and Truncate (1000 chars). Hard-fail if missing.
        lyric_path = os.path.join(lyrics_raw_path, base_name + ".txt")
        if not os.path.exists(lyric_path):
            raise FileNotFoundError(f"{filename}: no lyrics found at {lyric_path}")
        with open(lyric_path, 'r', encoding='utf-8') as f:
            clean_text = " ".join(f.read().split())[:1000]
        lyr_out = os.path.join(PROCESSED_DIR, "lyrics", f"{track_id}.txt")
        with open(lyr_out, 'w', encoding='utf-8') as f:
            f.write(clean_text)
        produced.append(lyr_out)

        # D. Match with the 7-column metadata. Hard-fail if the track has no label row.
        row = labels_df[labels_df['file_name'] == filename]
        if row.empty:
            raise KeyError(
                f"{filename} has no row in raw_data_labels.csv — fix the CSV, do not fallback."
            )

        data_log.append({
            "track_id": track_id,
            "file_name": filename,
            "song_title": row['song_title'].values[0],
            "artist_name": row['artist_name'].values[0],
            "genre": row['genre'].values[0],
            "emotion_category": row['emotion_category'].values[0],
            "emotion_subcategory": row['emotion_subcategory'].values[0],
            "contributor": row['contributor'].values[0],
            # Project-relative paths (downstream 02/03/04 consume these).
            "standardized_audio_path": f"01_Processed_Data/audio/{track_id}.wav",
            "standardized_image_path": f"01_Processed_Data/images/{track_id}.jpg",
            "lyrics_path":             f"01_Processed_Data/lyrics/{track_id}.txt",
        })

    except Exception:
        # Atomic cleanup: remove any files we wrote for this track before re-raising,
        # so a partial set of {track_id}.wav/.jpg/.txt never persists on disk.
        for p in produced:
            try:
                os.remove(p)
            except OSError:
                pass
        raise  # fail loudly — do not silently skip tracks (Directive 4: track_id alignment).

# --- STEP 4: SAVE THE MASTER MANIFEST + CONTRACT ASSERTIONS ---
final_metadata = pd.DataFrame(data_log)

REQUIRED_METADATA_COLS = [
    "track_id", "file_name", "song_title", "artist_name", "genre",
    "emotion_category", "emotion_subcategory", "contributor",
    "standardized_audio_path", "standardized_image_path", "lyrics_path",
]
missing = set(REQUIRED_METADATA_COLS) - set(final_metadata.columns)
assert not missing, f"metadata.csv missing required columns: {missing}"
assert final_metadata["track_id"].is_unique, "track_id is not unique"
assert final_metadata["emotion_category"].notna().all(), "missing emotion_category"
assert len(final_metadata) == len(files), (
    f"Row misalignment: metadata has {len(final_metadata)} rows, "
    f"processed {len(files)} input files"
)

final_metadata.to_csv(os.path.join(PROCESSED_DIR, "metadata.csv"), index=False)

print(f"\n✅ SUCCESS! {len(final_metadata)} tracks standardized with rich metadata.")
print(f"   Emotion categories present: {sorted(final_metadata['emotion_category'].unique())}")


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
📖 Loaded rich labels for 150 tracks.
🚀 Processing 150 tracks...

✅ SUCCESS! 150 tracks standardized with rich metadata.
   Emotion categories present: ['Awe, Beauty & Transcendence', 'Dreaminess, Curiosity & Ambiguity', 'Sadness, Loss & Lament', 'Tender & Intimacy', 'Tension, Threat & Aggression', 'Vital & Joyful']


In [ ]:
# --- VISUAL CHECKS (Directive 5) ---
# Waveform panel + thumbnail grid written to 04_Visual_Checks/01_{timestamp}/.
# Purpose: catch silent WAVs and corrupted JPGs at source, before any embedder sees them.
import random
import matplotlib.pyplot as plt
import numpy as np
from scipy.io import wavfile
from PIL import Image

# Archive the VIS_DIR contents from any previous run of notebook 01 before writing.
archive_previous_run(VIS_DIR, notebook_id=NOTEBOOK_ID)

stamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
vis_out = os.path.join(VIS_DIR, f"{NOTEBOOK_ID}_{stamp}")
os.makedirs(vis_out, exist_ok=True)

# A. Waveform panel — 12 random tracks, 3×4 grid.
random.seed(42)
track_ids_sorted = sorted(final_metadata["track_id"].tolist())
sample_ids = random.sample(track_ids_sorted, min(12, len(track_ids_sorted)))

rows, cols = 3, 4
fig, axes = plt.subplots(rows, cols, figsize=(16, 8))
for ax, tid in zip(axes.flat, sample_ids):
    wav_path = os.path.join(PROCESSED_DIR, "audio", f"{tid}.wav")
    sr, y = wavfile.read(wav_path)
    if y.ndim > 1:  # stereo -> mono for plotting only
        y = y.mean(axis=1)
    t = np.linspace(0, len(y) / sr, len(y))
    ax.plot(t, y, linewidth=0.5)
    ax.set_title(f"{tid}  |  {len(y)/sr:.1f}s  |  peak={int(np.abs(y).max())}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
# Hide any unused axes if fewer than 12 tracks.
for ax in axes.flat[len(sample_ids):]:
    ax.axis("off")
fig.suptitle("Waveform sanity check — notebook 01", fontsize=12)
fig.tight_layout()
fig.savefig(os.path.join(vis_out, "waveforms.png"), dpi=120)
plt.close(fig)

# B. Thumbnail grid — all tracks, 10×15 (for 150 tracks; auto-grid otherwise).
n = len(track_ids_sorted)
t_cols = 15
t_rows = int(np.ceil(n / t_cols))
fig, axes = plt.subplots(t_rows, t_cols, figsize=(22, max(2, 1.6 * t_rows)))
axes_flat = np.atleast_1d(axes).flat
for ax, tid in zip(axes_flat, track_ids_sorted):
    img_path = os.path.join(PROCESSED_DIR, "images", f"{tid}.jpg")
    ax.imshow(Image.open(img_path))
    ax.set_title(tid, fontsize=6)
    ax.axis("off")
for ax in list(axes_flat)[n:]:
    ax.axis("off")
fig.suptitle("Thumbnail sanity check — notebook 01", fontsize=12)
fig.tight_layout()
fig.savefig(os.path.join(vis_out, "thumbnails.png"), dpi=120)
plt.close(fig)

print(f"✅ Visual checks written to {vis_out}")


NameError: name 'archive_previous_run' is not defined